In [1]:
#import libraries
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
import sqlite3


In [2]:
#1. import database --> data 

In [3]:
conn = sqlite3.connect('customer_churn.db')

sql_query = """
SELECT name 
FROM sqlite_master
WHERE type ='table'
"""

tables = pd.read_sql(sql_query,conn)

#create dataframe for each table 

for table_name in tables['name']:
    df = pd.read_sql(f"SELECT * FROM {table_name}",conn)
    globals()[f"df_{table_name}"] = df
    print(f"Create dataframe: df_{table_name}")

Create dataframe: df_db_customer
Create dataframe: df_db_subscription
Create dataframe: df_db_support


In [4]:
df_db_customer.head()

,customerid,name,country,state,gender,dob,interests,pincode
0,0002-ORFBO,keshav,India,Maharashtra,Male,1982-04-12 00:00:00,travel,None
1,0003-MKNFE,raghav,India,Karnataka,Male,1995-11-23 00:00:00,None,None
2,0004-TLHLJ,lalita,India,Delhi,Female,1978-02-15 00:00:00,movie,None
3,0011-IGKFF,mohan,India,Nagaland,Male,2001-08-30 00:00:00,None,None
4,0013-EXCHZ,mira,India,Delhi,Female,1990-05-05 00:00:00,drama,None


In [5]:
df_db_customer.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   customerid  21 non-null     object
 1   name        21 non-null     object
 2   country     18 non-null     object
 3   state       21 non-null     object
 4   gender      21 non-null     object
 5   dob         21 non-null     object
 6   interests   4 non-null      object
 7   pincode     0 non-null      object
dtypes: object(8)
memory usage: 1.4+ KB


In [6]:
# rename column - name  as customer name 
# drop columns interest and pincode 
# change data type of dob column
# data standardization of gender
# fix missing values country


In [7]:
# rename columns name in customer_name
df_db_customer.rename(columns={'name' : 'customer_name'},inplace=True)

In [8]:
# drop columns - interest and pincode 
# on the bases of index 
#df_db_customer.drop(df_db_customer.columns[-2:],axis=1)
# on the bases of columns name 

df_db_customer.drop(columns=['interests','pincode'],inplace=True)

In [9]:
# change data type 

df_db_customer['dob']= pd.to_datetime(df_db_customer['dob'])

In [10]:
df_db_customer['gender'].unique()

array(['Male', 'Female', 'Women', 'Men'], dtype=object)

In [11]:
#replacing the values
df_db_customer['gender'].replace({'Men' : 'Male','Women' : 'Female'},inplace=True)

C:\Users\HP\AppData\Local\Temp\ipykernel_6576\161336515.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_db_customer['gender'].replace({'Men' : 'Male','Women' : 'Female'},inplace=True)


In [12]:
# fix missing values - country
df_db_customer[df_db_customer['country'].isna()]

,customerid,customer_name,country,state,gender,dob
5,0013-MHZWF,durga,None,Delhi,Female,1988-12-10
8,0015-UOCOJ,maya,None,Kathmandu,Female,1985-07-07
12,0018-NYROU,chitra,None,Telangana,Female,2004-12-01


In [13]:
state_country_mapping = df_db_customer.dropna(subset=['country']).set_index('state')['country'].to_dict()

In [14]:
df_db_customer['country']= df_db_customer['country'].fillna(df_db_customer['state'].map(state_country_mapping))

In [15]:
df_db_subscription.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,churn_score
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,None,None,13.99,627,12
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,91
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,None,None,6.99,210,34
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,None,None,22.99,1725,8
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,88


In [16]:
df_db_subscription.info()
#cltv customer live time value

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   customerid               21 non-null     object 
 1   subscription_start_date  21 non-null     object 
 2   subscription_type        21 non-null     object 
 3   renewal_date             21 non-null     object 
 4   plan_type                21 non-null     object 
 5   contract_type            21 non-null     object 
 6   cancellation_date        6 non-null      object 
 7   cancellation_reason      6 non-null      object 
 8   monthly_charges          21 non-null     float64
 9   cltv                     21 non-null     int64  
 10  churn_score              21 non-null     int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 1.9+ KB


In [17]:
# change data type of columns -- subscription_start_date , cancellation_date , renewal_date

In [18]:
# change multy_col date time in one 
date_col = ['subscription_start_date','cancellation_date','renewal_date']

df_db_subscription[date_col] = df_db_subscription[date_col].apply(pd.to_datetime)

In [19]:
df_db_subscription.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 11 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               21 non-null     object        
 1   subscription_start_date  21 non-null     datetime64[ns]
 2   subscription_type        21 non-null     object        
 3   renewal_date             21 non-null     datetime64[ns]
 4   plan_type                21 non-null     object        
 5   contract_type            21 non-null     object        
 6   cancellation_date        6 non-null      datetime64[ns]
 7   cancellation_reason      6 non-null      object        
 8   monthly_charges          21 non-null     float64       
 9   cltv                     21 non-null     int64         
 10  churn_score              21 non-null     int64         
dtypes: datetime64[ns](3), float64(1), int64(2), object(5)
memory usage: 1.9+ KB


In [20]:
df_db_support.head()

,customerid,complaint_date,escalations,csat_score,col_1,comment
0,0003-MKNFE,2024-08-28 00:00:00,N,60,None,service issue
1,0003-MKNFE,2024-08-28 00:00:00,Y,10,None,demaned refund
2,0013-EXCHZ,2024-01-20 00:00:00,Y,20,None,None
3,0013-MHZWF,2025-03-18 00:00:00,N,90,None,guidance to renew
4,0013-SMEOE,2024-11-01 00:00:00,N,30,None,None


In [21]:
df_db_support.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   customerid      9 non-null      object
 1   complaint_date  9 non-null      object
 2   escalations     9 non-null      object
 3   csat_score      9 non-null      int64 
 4   col_1           0 non-null      object
 5   comment         4 non-null      object
dtypes: int64(1), object(5)
memory usage: 564.0+ bytes


In [22]:
df_db_support['complaint_date'] = pd.to_datetime(df_db_support['complaint_date'])

In [23]:
df_db_support.drop(columns=['col_1'],inplace=True)

# feacture engineering and data analysis

In [24]:
df_db_subscription['churn_flag'] = np.where(df_db_subscription['cancellation_date'].notna(),1,0)

In [25]:
df_db_support.head(10)

,customerid,complaint_date,escalations,csat_score,comment
0,0003-MKNFE,2024-08-28,N,60,service issue
1,0003-MKNFE,2024-08-28,Y,10,demaned refund
2,0013-EXCHZ,2024-01-20,Y,20,None
3,0013-MHZWF,2025-03-18,N,90,guidance to renew
4,0013-SMEOE,2024-11-01,N,30,None
5,0017-IUDMW,2024-04-10,Y,25,None
6,0019-EFAEP,2024-09-27,Y,30,None
7,0022-TCJCI,2024-09-13,Y,10,None
8,0022-TCJCI,2024-09-14,N,90,received refund


In [26]:
#removing the duplicate
df_db_support.drop_duplicates(subset=['customerid'],keep='last',inplace=True)

In [27]:
# joinning the table 

df = (df_db_subscription.merge(df_db_customer,on= 'customerid',how= 'left').merge(df_db_support,on='customerid',how= 'left'))


In [28]:
df.shape

(21, 21)

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 21 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   customerid               21 non-null     object        
 1   subscription_start_date  21 non-null     datetime64[ns]
 2   subscription_type        21 non-null     object        
 3   renewal_date             21 non-null     datetime64[ns]
 4   plan_type                21 non-null     object        
 5   contract_type            21 non-null     object        
 6   cancellation_date        6 non-null      datetime64[ns]
 7   cancellation_reason      6 non-null      object        
 8   monthly_charges          21 non-null     float64       
 9   cltv                     21 non-null     int64         
 10  churn_score              21 non-null     int64         
 11  churn_flag               21 non-null     int64         
 12  customer_name            21 non-null  

In [30]:
df.to_csv('exported_chrun_data.csv',index=False)

In [31]:
df.columns

Index(['customerid', 'subscription_start_date', 'subscription_type',
       'renewal_date', 'plan_type', 'contract_type', 'cancellation_date',
       'cancellation_reason', 'monthly_charges', 'cltv', 'churn_score',
       'churn_flag', 'customer_name', 'country', 'state', 'gender', 'dob',
       'complaint_date', 'escalations', 'csat_score', 'comment'],
      dtype='object')

data analysis

In [32]:
churn_rate = df['churn_flag'].mean()*100
print("churn rate = ",round(churn_rate,2),"%")

churn rate =  28.57 %


In [33]:
# retenion rate 
retention_rate = 100-churn_rate
print("retention rate = ",round(retention_rate,2),"%")

retention rate =  71.43 %


In [34]:
df.head()

,customerid,subscription_start_date,subscription_type,renewal_date,plan_type,contract_type,cancellation_date,cancellation_reason,monthly_charges,cltv,...,churn_flag,customer_name,country,state,gender,dob,complaint_date,escalations,csat_score,comment
0,0002-ORFBO,2021-03-15,Refferal,2025-03-15,Standard,Annual,NaT,None,13.99,627,...,0,keshav,India,Maharashtra,Male,1982-04-12,NaT,NaN,NaN,NaN
1,0003-MKNFE,2020-08-01,Paid,2024-08-01,Premium,Annual,2024-09-10,Switched to competitor,12.99,1150,...,1,raghav,India,Karnataka,Male,1995-11-23,2024-08-28,Y,10.0,demaned refund
2,0004-TLHLJ,2022-11-20,Organic,2025-11-20,Basic,Monthly,NaT,None,6.99,210,...,0,lalita,India,Delhi,Female,1978-02-15,NaT,NaN,NaN,NaN
3,0011-IGKFF,2019-05-10,Paid,2025-05-10,Premium,Annual,NaT,None,22.99,1725,...,0,mohan,India,Nagaland,Male,2001-08-30,NaT,NaN,NaN,NaN
4,0013-EXCHZ,2023-01-05,Refferal,2024-01-05,Standard,Monthly,2024-02-28,Too expensive,13.99,195,...,1,mira,India,Delhi,Female,1990-05-05,2024-01-20,Y,20.0,None


In [35]:
# chrun by plan type 
churn_by_plan =(df.groupby('plan_type')['churn_flag'].mean().mul(100).round(2).reset_index(name= 'churn_rate_pct'))
print(churn_by_plan)

  plan_type  churn_rate_pct
0     Basic           60.00
1   Premium           14.29
2  Standard           22.22


In [36]:
churn_by_conttype = df.groupby(['plan_type','contract_type'])['churn_flag'].mean()*100
print(churn_by_conttype)

plan_type  contract_type
Basic      Annual            0.000000
           Monthly          75.000000
Premium    Annual           14.285714
Standard   Annual            0.000000
           Monthly          40.000000
Name: churn_flag, dtype: float64


In [37]:
retention_rate_by_pc = 100-(df.groupby(['plan_type','contract_type'])['churn_flag'].mean()*100)
print(retention_rate_by_pc)

plan_type  contract_type
Basic      Annual           100.000000
           Monthly           25.000000
Premium    Annual            85.714286
Standard   Annual           100.000000
           Monthly           60.000000
Name: churn_flag, dtype: float64


In [44]:
state_revenue = df.groupby('state')['monthly_charges'].sum().reset_index(name="total_monthly_charge")
print(state_revenue)

           state  total_monthly_charge
0          Delhi                 52.96
1      Karnataka                 20.98
2      Kathmandu                 20.98
3    Maharashtra                 50.97
4      Meghalaya                 42.97
5       Nagaland                 22.99
6      Rajasthan                 36.98
7      Telangana                 30.98
8  Uttar Pradesh                115.98


In [52]:
state_churn_revenue = df.groupby('state').agg( total_sales = ('monthly_charges','sum'),churn_rate =(('churn_flag',lambda x: x.mean() * 100)))
print(state_churn_revenue)

               total_sales  churn_rate
state                                 
Delhi                52.96   25.000000
Karnataka            20.98  100.000000
Kathmandu            20.98    0.000000
Maharashtra          50.97    0.000000
Meghalaya            42.97   66.666667
Nagaland             22.99    0.000000
Rajasthan            36.98    0.000000
Telangana            30.98   50.000000
Uttar Pradesh       115.98    0.000000
